<a href="https://colab.research.google.com/github/ScottG619/Advanced-Computer-Programming-with-Generative-AI/blob/Class-Activities/Week3_Lab_Activities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 Lab — Encapsulation (OOP II)

**192-201 Advanced Computer Programming with Generative AI**

Open in **Google Colab**: *Runtime → Run all* works. Cells that are *meant* to fail catch the error and print it, so the notebook does not stop.

Reference: *Think Python, 3rd Edition* — Chapter 15 (`__init__`) + lecturer supplement on encapsulation and `@property`.
Builds on **Week 2** (Classes & Objects). Practice classes continue from *Python 100 Programs* (Devbrat Rudra), **SET-2 #10** and **SET-10 #8**.

---

### About this lab

This is a **guided practice notebook** — it is **not graded**. Last week you built classes with attributes, methods, and `__str__`. This week you make them **safe**: an object should **never** hold invalid data (a negative balance, a negative age). That idea is **encapsulation**.

Work through the parts **in order**. For each part:
1. Read the short guidance.
2. **Run every code cell** and look at the result.
3. Complete each **Your turn** / **TODO** task.

> **Run a cell:** click it, then press **Shift + Enter** (or the play button on the left).

> **AI-use policy — Level 1 (explain only):** you may ask an AI assistant to *explain* an idea, but **type all the code yourself**. Do not paste AI-written code today.

## Part 0 — Recap: last week's class

Here is the `BankAccount` from Week 2. Run it — it works, but it has a hidden weakness we will fix today.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

    def __str__(self):
        return f'{self.owner}: ${self.balance}'


acct = BankAccount('Mai', 100)
acct.deposit(50)
print(acct)  # Mai: $150

Mai: $-899


## Part 1 — The problem: nothing stops invalid data

Because `balance` is a plain public attribute, **anyone** can set it to nonsense. Run this and see that Python happily allows an impossible balance:

In [ ]:
acct = BankAccount('Mai', 100)
acct.balance = -9999  # this should NEVER be allowed!
print(acct)           # Mai: $-9999  <- broken state

> The object is now in an **invalid state**. Nothing checked the value. **Encapsulation** is the fix: we keep the data *internal* and force all changes through code that can **validate** them.

**Your turn.** In one sentence (edit this Markdown cell), write down another impossible value a bank account should reject.

*Your answer:* The output should not be lower than 0 and it should display as "The amount cannot be lower than 0" or something like that.

## Part 2 — Start valid: validate inside `__init__`

The first job of the constructor `__init__` is to build the object in a **valid initial state**. We check the value there and refuse to build a bad object by raising an error.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        if balance < 0:
            raise ValueError('balance cannot be negative')
        self.owner = owner
        self.balance = balance


good = BankAccount('Mai', 100)  # fine
print(good.balance)             # 100

100


Now try to build an **invalid** account. This **should** raise a `ValueError` (that is the point). The `try` / `except` lets Colab keep running after the error.

In [ ]:
try:
    bad = BankAccount('Mai', -50)
except ValueError as e:
    print(type(e).__name__ + ':', e)  # expected: ValueError: balance cannot be negative

ValueError: balance cannot be negative


> Raising an error early is good: the broken object is never created. But `__init__` only protects the *starting* value — someone could still write `acct.balance = -50` afterwards. Parts 3–5 close that gap.

## Part 3 — Hide the data: private attributes

To stop direct changes, we mark an attribute as **internal** by starting its name with an underscore. A **double** underscore (`__balance`) makes Python hide the name so it cannot be read or set from outside in the usual way *(W3Schools: Encapsulation)*.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        if balance < 0:
            raise ValueError('balance cannot be negative')
        self.owner = owner
        self.__balance = balance  # private: note the __ prefix


acct = BankAccount('Mai', 100)
print(acct.owner)  # Mai  (public, fine)

try:
    print(acct.__balance)
except AttributeError as e:
    print(type(e).__name__ + ':', e)  # expected: AttributeError — it is hidden

Mai
AttributeError: 'BankAccount' object has no attribute '__balance'


> The last line **fails on purpose** — that is encapsulation working. The balance is now protected from casual `acct.__balance = -50` mistakes. Next we need a *safe* way to read and change it — the getter and setter.

## Part 4 — Controlled access: getter and setter

A **getter** returns the private value; a **setter** changes it *after checking it is valid*. This is the W3Schools encapsulation pattern (`get_age` / `set_age` with an `if` check).

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.set_balance(balance)  # reuse the setter's validation

    def get_balance(self):  # getter
        return self.__balance

    def set_balance(self, amount):  # setter WITH validation
        if amount < 0:
            raise ValueError('balance cannot be negative')
        self.__balance = amount


acct = BankAccount('Mai', 100)
print(acct.get_balance())  # 100
acct.set_balance(120)      # allowed
print(acct.get_balance())  # 120

Now the guard cannot be bypassed. Run this to confirm a bad value is rejected:

In [ ]:
try:
    acct.set_balance(-5)
except ValueError as e:
    print(type(e).__name__ + ':', e)  # expected: ValueError: balance cannot be negative

**Your turn.** Add a `deposit(self, amount)` method that uses `set_balance` to add money *(so the validation still applies)*. Then test it with a deposit.

Type this yourself (Level 1). Replace `pass` below.

In [ ]:
# Your code here — type it yourself.
# Hint: define BankAccount again, copy Parts 4 methods, then add:
#   def deposit(self, amount):
#       ...
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.set_balance(balance)  # reuse the setter's validation

    def get_balance(self):  # getter
        return self.__balance

    def set_balance(self, amount):  # setter WITH validation
        if amount < 0:
            raise ValueError('balance cannot be negative')
        self.__balance = amount

    def deposit(self, amount):
        self.set_balance(self.get_balance() + amount)
        return self.get_balance()


acct = BankAccount('Mai', 200)
print(acct.get_balance())  # 100
acct.set_balance(120)      # allowed
print(acct.get_balance())  # 120
acct.deposit(100)
print(acct.get_balance())

200
120
220


## Part 5 — The Pythonic way: `@property`

Getters and setters work, but Python programmers prefer `@property`. It gives the **safety of a setter** with the **clean look of a normal attribute** — you write `acct.balance` and `acct.balance = 120`, and the validation still runs behind the scenes.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance  # this calls the setter below

    @property
    def balance(self):  # the getter
        return self.__balance

    @balance.setter
    def balance(self, amount):  # the setter (validates)
        if amount < 0:
            raise ValueError('balance cannot be negative')
        self.__balance = amount


acct = BankAccount('Mai', 100)
print(acct.balance)  # 100  <- looks like a normal attribute
acct.balance = 120   # runs the setter -> validated
print(acct.balance)  # 120

Best of both worlds: clean syntax **and** protection. Confirm the guard still fires:

In [ ]:
try:
    acct.balance = -1
except ValueError as e:
    print(type(e).__name__ + ':', e)  # expected: ValueError: balance cannot be negative

## Part 6 — Guided build: a bullet-proof `Student`

Take the `Student` class idea from Week 2 and make it **impossible to break**. Fill in the `# TODO` lines using `@property`.

**Rules the object must enforce**
- `name` must not be empty.
- `age` must be between 1 and 120.
- Setting either to an invalid value must raise `ValueError`.

Type the TODOs yourself (Level 1). After that, uncomment the checks at the bottom.

In [ ]:
class Student:
    def __init__(self, name, age):
        self.name = name  # calls the name setter
        self.age = age    # calls the age setter

    @property
    def name(self):
        return self.__name

    @name.setter
    def name(self, name):
        if name == '':
            raise ValueError('Name cannot be empty')
        self.__name = name
        pass

    @property
    def age(self):
        return self.__age

    @age.setter
    def age(self, age):
        if age < 1 or age > 120:
            raise ValueError('Age must be between 1 and 120')
        self.__age = age
        pass

    def __str__(self):
        return f'{self.name} (age {self.age})'


# After completing the TODOs, uncomment these checks:
s = Student('Mai', 19)
print(s)                 # Mai (age 19)
try:
    s.age = 200          # should raise ValueError
except ValueError as e:
    print(type(e).__name__ + ':', e)
try:
    Student('', 19)      # should raise ValueError
except ValueError as e:
    print(type(e).__name__ + ':', e)

Mai (age 19)
ValueError: Age must be between 1 and 120
ValueError: Name cannot be empty


**Check your work.** Once the TODOs are done: `Student('Mai', 19)` builds fine and prints `Mai (age 19)`; `s.age = 200` raises `ValueError`; and `Student('', 19)` raises `ValueError`.

## Part 7 — Design review (work in pairs)

With a partner, look at each other's `Student` class and fill in this table for **one** attribute you protected.

**Example — BankAccount.balance:**

| Attribute | Rule it enforces | What happens on bad input |
|---|---|---|
| `balance` | must be >= 0 | raises `ValueError` |

**Your turn.** Double-click the table below and fill it in for your class:

| Attribute | Rule it enforces | What happens on bad input |
|---|---|---|
| *name* | *cannot be empty* | *ValueError: Name cannot be empty* |
| *age* | *must be between 1 and 120* | *ValueError: Age must be between 1 and 120* |

## Part 8 — Learning with an AI assistant (Level 1)

You may ask an assistant to **explain**, not to write your validation. Good Level-1 questions:

- "Explain what the `@property` decorator does, with a simple example."
- "Why does a double underscore `__balance` make an attribute hard to access from outside?"
- "What is the difference between a getter/setter and a plain attribute?"
- "When should code `raise ValueError` instead of `return`?"

**Not allowed today:** asking the assistant to write the validated `Student` class for you and pasting it in.

## Before you finish — quick checklist

You should now be able to:

- [ ] Explain why a public attribute can leave an object in an **invalid state**
- [ ] Validate a value inside `__init__` so an object starts valid
- [ ] Make an attribute **private** with a `__` prefix
- [ ] Write a **getter** and a **setter** that validates before storing
- [ ] Use `@property` to get clean syntax **and** validation
- [ ] Complete the bullet-proof `Student` class

**Before Week 4:** make sure your validated `Student` runs and is pushed to your GitHub repo. Next week we add **inheritance & polymorphism** — specialised classes (e.g. `Shape` → `Circle`, `Rectangle`) that share and override behaviour.

*Well done!*